# SHYFEM Antsiranana Bay Analysis

Explore the SHYFEM ocean model results for Antsiranana Bay, Madagascar (March 31 - April 5, 2021)

In [1]:
import numpy as np
import xarray as xr
import xugrid as xu
import hvplot.xugrid
import holoviews as hv
import panel as pn

pn.extension()
hv.extension('bokeh')

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
import os
local_file = "surf.ous.nc"

if os.path.exists(local_file):
    ds = xr.open_dataset(local_file)
else:
    ds = xr.open_dataset('https://r2-pub.openscicomp.io/protocoast/testing/surf.ous.nc')

print(f"Variables: {list(ds.data_vars)}")

Variables: ['longitude', 'latitude', 'total_depth', 'element_index', 'topology', 'water_level', 'u_velocity', 'v_velocity']


## Add UGRID Metadata

In [3]:
def add_shyfem_ugrid_metadata(ds):
    """Add UGRID conventions metadata to SHYFEM dataset"""
    mesh_name = "mesh_topology"
    
    mesh_attrs = {
        "cf_role": "mesh_topology",
        "topology_dimension": 2,
        "node_coordinates": "longitude latitude",
        "face_node_connectivity": "element_index",
        "face_dimension": "element"
    }
    
    # Add mesh topology variable
    ds = ds.assign({mesh_name: xr.DataArray(0, attrs=mesh_attrs)})
    
    # Update element_index attributes
    ds.element_index.attrs.update({
        "cf_role": "face_node_connectivity",
        "start_index": 1  # SHYFEM uses 1-based indexing
    })
    
    # Add mesh attribute to data variables
    for var in ds.data_vars:
        if "node" in ds[var].dims:
            ds[var].attrs["mesh"] = mesh_name
            ds[var].attrs["location"] = "node"
    
    return ds

ds_ugrid = add_shyfem_ugrid_metadata(ds)
print("Added UGRID metadata")

Added UGRID metadata


## Convert to UgridDataset

In [4]:
%%time
uds = xu.UgridDataset(ds_ugrid)
print(f"Created UgridDataset with {len(uds.data_vars)} variables")

Created UgridDataset with 7 variables
CPU times: user 46.2 ms, sys: 3.97 ms, total: 50.2 ms
Wall time: 49.6 ms


## Water Level Interactive Map

In [5]:
%%time
uds['water_level'].isel(time=-1).hvplot.trimesh(
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='turbo',
    colorbar=True,
    width=600,
    height=400,
)

CPU times: user 1.82 s, sys: 67.1 ms, total: 1.89 s
Wall time: 1.9 s


:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [x,y]   (x_y z)

## Current Speed Interactive Map

In [6]:
%%time
# Surface velocity (level=0)
u = uds['u_velocity'].isel(time=-1, level=0)
v = uds['v_velocity'].isel(time=-1, level=0)
speed = np.sqrt(u**2 + v**2)

speed.hvplot.trimesh(
    rasterize=True,
    geo=True,
    tiles='OSM',
    cmap='turbo',
    colorbar=True,
    width=600,
    height=400,
)

CPU times: user 71.5 ms, sys: 3.02 ms, total: 74.5 ms
Wall time: 73.9 ms


:DynamicMap   []
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [x,y]   (x_y z)